# `msep` Quickstart — Multi-Scale Entropy Profiling in 5 Minutes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aistanbulresearch/msep/blob/main/notebooks/quickstart/quickstart_colab.ipynb)

This notebook demonstrates the **multi-scale entropy profiling** (MSEP) framework on a small synthetic dataset bundled with the `msep` package. It reproduces the paper's qualitative **"individually diverse, collectively disciplined"** paradox in under 5 minutes — no data download required, no GPU needed.

What you will see:

1. Load the bundled synthetic AnnData via `msep.datasets.load_example()`
2. Run the full MSEP pipeline with a single call to `msep.profile(...)`
3. Inspect the **paradox summary** — per-cell entropy vs. across-cells CV
4. Visualise the **pathway-CV heatmap** and the **paradox scatter**
5. Compute the **Coordination Score** — a single-number summary

**Reference:** Çavuş & Kuşkucu (2026). *Multi-Scale Entropy Profiling Reveals Individually Diverse but Collectively Disciplined Transcriptional Programs in Chordoma Stem Cells.* Submitted to Genome Biology.

**Package:** https://github.com/aistanbulresearch/msep · `pip install msep`

## 1. Install `msep`

Only one line is required. The `[plotting]` extra pulls in matplotlib + seaborn for the publication-style figures.

> **About the pandas warning:** if Colab prints `google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.`, the message is cosmetic — pandas 2.3.x is fully backwards-compatible with this package's usage. The `--upgrade-strategy only-if-needed` flag below tells pip to leave Colab's pre-installed pandas in place, silencing the warning.

In [ ]:
!pip install -q --upgrade-strategy only-if-needed 'msep[plotting]>=1.1.0'

## 2. Load the bundled demo dataset

`load_example()` returns a 500-cell synthetic AnnData (four cell types, four cancer-defense pathways) designed to reproduce the paper's qualitative paradox. The data is generated on-the-fly from a seeded RNG, so nothing is downloaded.

In [ ]:
import msep

adata = msep.datasets.load_example()
print(adata)
print('\nCell-type composition:')
print(adata.obs['cell_type'].value_counts())

## 3. Run the full MSEP pipeline

A single call to `msep.profile()` computes:

- per-cell Shannon entropy (global + per-pathway)
- across-cells coefficient of variation (CV) per (cell type, pathway)
- per-gene CV for each pathway
- XBP1 stress-consolidation analysis on the CSC compartment

Expected runtime on Colab's free CPU: **< 10 seconds**.

In [ ]:
result = msep.profile(
    adata,
    pathways='cancer_defense',
    cell_type_key='cell_type',
    compute_bootstrap=False,       # skip for speed; enable for paper-grade CIs
    compute_gene_cv=True,
    compute_xbp1=True,
    perturbation_cell_type='CSC_TBXT+',
)
result

## 4. The multi-scale paradox

`result.paradox_summary` is a one-row-per-cell-type view combining per-cell entropy with across-cells CV per pathway. Look at `CSC_TBXT+`:

- It has the **highest** median per-cell entropy → each individual CSC is transcriptomically versatile
- It has the **lowest** `cv_emt` → all CSC converge on the same EMT programme
- It has the **highest** `cv_immune_evasion` → immune evasion expression is heterogeneous across cells

This is the signature "individually diverse, collectively disciplined" phenotype the paper introduces.

In [ ]:
result.paradox_summary.round(3)

## 5. Pathway-level CV heatmap

A visual of `result.pathway_cv` — rows are cell types, columns are pathways, colour encodes population-level heterogeneity. Lower = more tightly coordinated.

In [ ]:
import matplotlib.pyplot as plt

fig = msep.plot_pathway_cv_heatmap(result, figsize=(6, 4))
plt.tight_layout()
plt.show()

## 6. Paradox scatter (entropy vs. CV)

The scatter places each cell type in the (per-cell entropy, across-cells EMT CV) plane. The paradox quadrant — **top left, high entropy + low CV** — is where `CSC_TBXT+` should land.

In [ ]:
fig = msep.plot_paradox(result, cv_pathway='emt', figsize=(6, 5))
plt.tight_layout()
plt.show()

## 7. Coordination Score — single-number summary

`result.coordination_scores` condenses the two axes into a single score per (cell type × pathway). Values near **1.0** indicate a strong paradox (high entropy + low CV); values near **0.0** indicate no paradox.

Rows where `classification == 'strong paradox'` are the ones to flag for follow-up biology.

In [ ]:
scores = result.coordination_scores
scores[scores['classification'] == 'strong paradox']

## 8. XBP1 stress consolidation

`result.xbp1` compares pathway CV between XBP1-zero, XBP1-low, and XBP1-high CSC cells. In the real chordoma data, XBP1-high cells tighten coordination across **all three** defence pathways simultaneously — a phenomenon that extends to five of nine cancer types tested in the paper. On the synthetic demo, you should see a similar direction of effect (XBP1-high CV < XBP1-zero CV).

In [ ]:
result.xbp1.round(3)

In [ ]:
result.consolidation_score()

## What next?

| Goal | Notebook |
|------|----------|
| Reproduce individual paper figures | [`notebooks/figures/`](../figures/) |
| Run synthetic ground-truth validation (Splatter) | [`notebooks/validation/`](../validation/) |
| Extend to 20 cancer types via CellxGene Census | [`notebooks/pan_cancer/`](../pan_cancer/) |
| Full chordoma analysis (106,584 cells) | [`examples/chordoma_msep.ipynb`](../../examples/chordoma_msep.ipynb) |

### Bring your own data

```python
import scanpy as sc, msep
adata = sc.read_h5ad('my_data.h5ad')
result = msep.profile(adata, pathways='cancer_defense', cell_type_key='cell_type')
result.paradox_summary
```

See the [package README](https://github.com/aistanbulresearch/msep#readme) for custom pathway definitions, MSigDB access (33,000+ gene sets), and Bayesian validation via scVI.